Cellule 1 : clone + install + données

In [ ]:
!git clone https://github.com/Hdlmrbt/3d-generation-reconstruction.git
%cd 3d-generation-reconstruction
!pip install -q trimesh pyyaml tqdm
!mkdir -p data/raw
!wget -q http://3dvision.princeton.edu/projects/2014/3DShapeNets/ModelNet10.zip -P data/raw
!unzip -q data/raw/ModelNet10.zip -d data/raw

Cellule 2 : cache des nuages de points

In [ ]:
from src.dataset import build_cache
CLASSES = ["chair", "table", "toilet"]
for s in ["train", "test"]:
    build_cache("data/raw/ModelNet10", CLASSES, s, n=1024, max_per_class=300)

Cellule 3 : entraînement du VAE

In [ ]:
import yaml, torch, pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader
from src.dataset import PointCloudDataset
from src.models import PointNetAE, PointNetVAE
from src.losses import chamfer

cfg = yaml.safe_load(open("configs/vae.yaml"))
torch.manual_seed(cfg["seed"])
dev = "cuda" if torch.cuda.is_available() else "cpu"

tr = DataLoader(PointCloudDataset("data/processed/train_1024.npz"), cfg["batch_size"], shuffle=True, drop_last=True)
te = DataLoader(PointCloudDataset("data/processed/test_1024.npz"), cfg["batch_size"])

vae = PointNetVAE(cfg["latent"], cfg["n_points"]).to(dev)
opt = torch.optim.Adam(vae.parameters(), lr=cfg["lr"])
hist = []

for ep in range(cfg["epochs"]):
    vae.train(); r_tr = kl_tr = 0
    for x, _ in tr:
        x = x.to(dev)
        rec, mu, logvar = vae(x)
        r = chamfer(rec, x).mean()
        kl = (-0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(1)).mean()
        loss = r + cfg["beta"] * kl
        opt.zero_grad(); loss.backward(); opt.step()
        r_tr += r.item(); kl_tr += kl.item()
    vae.eval(); r_te = 0
    with torch.no_grad():
        for x, _ in te:
            x = x.to(dev)
            mu, _ = vae.encode(x)
            r_te += chamfer(vae.decode(mu), x).mean().item()
    hist.append((ep, r_tr / len(tr), kl_tr / len(tr), r_te / len(te)))
    if ep % 10 == 0:
        print(hist[-1])

Path("checkpoints").mkdir(exist_ok=True)
torch.save(vae.state_dict(), "checkpoints/vae.pt")
pd.DataFrame(hist, columns=["epoch", "train_cd", "train_kl", "test_cd"]).to_csv("results/tables/03_training_log.csv", index=False)

Cellule 3 : courbe de loss

In [ ]:
import matplotlib.pyplot as plt
h = pd.read_csv("results/tables/03_training_log.csv")
plt.plot(h.epoch, h.train_cd, label="train")
plt.plot(h.epoch, h.test_cd, label="test")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("Chamfer"); plt.legend()
plt.savefig("results/figures/03_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

Cellule 5 : recharger la baseline

In [ ]:
from google.colab import files
files.upload()          # choisis checkpoints/baseline_ae.pt sur ton PC
ae = PointNetAE(128, 1024).to(dev)
ae.load_state_dict(torch.load("baseline_ae.pt", map_location=dev))
ae.eval()

cellules 6 : métriques et comparaison

In [ ]:
from src.metrics import mmd_cov

ref = torch.cat([x for x, _ in te]).to(dev)       # tout le jeu de test
n = len(ref)

# Génération VAE : z ~ N(0, I)
vae.eval()
gen_vae = vae.sample(n, dev)

# Génération baseline : gaussienne ajustée sur les codes du train
with torch.no_grad():
    Z = torch.cat([ae.encode(a.to(dev)) for a, _ in tr]).cpu()
mu_z = Z.mean(0)
cov_z = torch.cov(Z.T) + 1e-4 * torch.eye(Z.shape[1])
z = torch.distributions.MultivariateNormal(mu_z, cov_z).sample((n,)).to(dev)
with torch.no_grad():
    gen_ae = ae.decode(z)

# Chamfer de reconstruction (test)
with torch.no_grad():
    cd_ae = torch.cat([chamfer(ae(a.to(dev))[0], a.to(dev)) for a, _ in te]).mean().item()
    cd_vae = torch.cat([chamfer(vae.decode(vae.encode(a.to(dev))[0]), a.to(dev)) for a, _ in te]).mean().item()

mmd_ae, cov_ae = mmd_cov(gen_ae, ref)
mmd_vae, cov_vae = mmd_cov(gen_vae, ref)

res = pd.DataFrame([
    {"model": "PointNet-AE + gaussienne", "test_chamfer": cd_ae, "MMD": mmd_ae, "COV": cov_ae},
    {"model": "PointNet-VAE", "test_chamfer": cd_vae, "MMD": mmd_vae, "COV": cov_vae},
])
res.to_csv("results/tables/03_comparison.csv", index=False)
print(res)

Cellule 7 : exemples visuels (générations VAE + interpolation)

In [ ]:
# Générations VAE
fig = plt.figure(figsize=(12, 4))
g = gen_vae[:12].cpu().numpy()
for i in range(12):
    ax = fig.add_subplot(2, 6, i + 1, projection="3d")
    ax.scatter(g[i, :, 0], g[i, :, 1], g[i, :, 2], s=1); ax.set_axis_off()
fig.suptitle("Générations du VAE")
fig.savefig("results/figures/03_generated_vae.png", dpi=150, bbox_inches="tight")

# Interpolation entre deux objets du test
ds = PointCloudDataset("data/processed/test_1024.npz")
i1 = int((ds.y == 0).nonzero()[0]); i2 = int((ds.y == 1).nonzero()[0])   # une chaise, une table
with torch.no_grad():
    z1 = vae.encode(ds[i1][0][None].to(dev))[0]
    z2 = vae.encode(ds[i2][0][None].to(dev))[0]
    steps = [vae.decode((1 - t) * z1 + t * z2)[0].cpu().numpy() for t in torch.linspace(0, 1, 6)]
fig = plt.figure(figsize=(12, 2.5))
for i, p in enumerate(steps):
    ax = fig.add_subplot(1, 6, i + 1, projection="3d")
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], s=1); ax.set_axis_off()
fig.suptitle("Interpolation chaise → table")
fig.savefig("results/figures/03_interpolation.png", dpi=150, bbox_inches="tight")
plt.show()

Récuperer et pousser

In [ ]:
!zip -r results_03.zip results/
from google.colab import files
files.download("results_03.zip")
files.download("checkpoints/vae.pt")